# DDNM inverse experiments

This notebook clones the repo, downloads one validation shard and the score-model checkpoint, then runs
**DDNM** and **DDNM + divergence-free physics** experiments for four observation operators:

- **sparse_grid** — velocity known at every 4th grid point (stride 4).
- **box_mask** — velocity known inside the central 32×32 window.
- **downsample** — average-pooled low-resolution observation 64×64 → 16×16.
- **blur** — periodic Gaussian blur (σ=2) with optional Gaussian observation noise.

DDNM corrects the predicted x̂_0 at each reverse step via the null-space formula:
  Hard DDNM (soft_lambda=0): x̂_0^corr = A†y + (I − A†A) x̂_0
  Soft DDNM (soft_lambda>0): x̂_0^corr = x̂_0 + λ A†(y − A x̂_0)
For MaskOperator both reduce to: M ⊙ y + (1 − M) ⊙ x̂_0.

Physics variant: after the data-consistency correction, x̂_0^corr is projected onto the
divergence-free subspace via Helmholtz FFT projection. For mask operators known pixels
are re-pasted so the observed region is preserved exactly.

Reproducibility convention: `VISUALIZATION_SAMPLE_COUNT` cases are selected with `VISUALIZATION_SEED`
and placed first, so PNG visualizations stay identical when different people use different metric
sample counts.

In [ ]:
#@title 1. Clone repo and install dependencies
from pathlib import Path
import os

REPO_URL = "https://github.com/SadreevAmir/DL_final_project.git"  #@param {type:"string"}
REPO_DIR = Path("/content/DL_final_project")

if not REPO_DIR.exists():
    !git clone -b RePaint --single-branch "$REPO_URL" "$REPO_DIR"
else:
    %cd "$REPO_DIR"
    !git pull --ff-only

%cd "$REPO_DIR"
!pip install -q -r requirements.txt

In [ ]:
#@title 2. Experiment settings
from pathlib import Path

DATA_SOURCE = "https://disk.yandex.ru/d/rrjDGzzX5cfFnA"  #@param {type:"string"}
EMA_CHECKPOINT_NAME = "best_score_vp_kolmogorov_velocity_256_to_64_coords_2ch_64x64_coords_bf16_20260517_221201_epoch_0014_val_0.028523_ema_model_only.pt"  #@param {type:"string"}
FULL_CHECKPOINT_NAME = "best_score_vp_kolmogorov_velocity_256_to_64_coords_2ch_64x64_coords_bf16_20260517_221201_epoch_0014_val_0.028523.pt"  #@param {type:"string"}
STATS_FILENAME = "kolmogorov_velocity_256_to_64_train_stats.json"  #@param {type:"string"}
VAL_SHARD_NAME = "val_000.npz"  #@param {type:"string"}

# Metrics use this many validation samples from the downloaded val shard.
METRIC_SAMPLE_COUNT = 16  #@param {type:"integer"}
SAMPLE_SEED = 0  #@param {type:"integer"}

# Keep these fixed across people/runs to make visualizations identical.
VISUALIZATION_SAMPLE_COUNT = 8  #@param {type:"integer"}
VISUALIZATION_SEED = 0  #@param {type:"integer"}
MAX_VISUALIZATIONS = VISUALIZATION_SAMPLE_COUNT

# Keep this fixed for the same corruption noise.
CORRUPTION_SEED = 0  #@param {type:"integer"}
NOISE_SIGMA = 0.0  #@param {type:"number"}
# Noise sigma for the blur operator specifically (blur observations are noisy by design).
BLUR_NOISE_SIGMA = 0.05  #@param {type:"number"}

# DDNM diffusion steps. Start with 64 for debug, use 128-256 for final runs.
DDNM_STEPS = 128  #@param {type:"integer"}

# soft_lambda=0 → hard DDNM (x̂_0^corr = A†y + (I−A†A)x̂_0).
# soft_lambda>0 → soft DDNM (x̂_0^corr = x̂_0 + λ A†(y − Ax̂_0)).
# Hard DDNM is recommended for mask/downsample; soft DDNM can be more stable for noisy blur.
SOFT_LAMBDA = 0.0  #@param {type:"number"}

# A100 default. Reduce to 16 or 8 if CUDA OOM.
RUN_BATCH_SIZE = 32  #@param {type:"integer"}
CASE_BATCH_SIZE = 256  #@param {type:"integer"}
CASE_BUILD_DEVICE = "cuda"  #@param ["cpu", "cuda"]
DDNM_SEED = 0  #@param {type:"integer"}

# Physics weight for DDNM + physics variant.
# Any value > 0 enables Helmholtz divergence-free projection on x̂_0^corr.
# The actual magnitude does not matter — set to 1.0 to enable, 0.0 to disable.
PHYSICS_DIV_WEIGHT = 1.0  #@param {type:"number"}

DOWNLOAD_FULL_CHECKPOINT_FOR_STATS = True  #@param {type:"boolean"}
DELETE_FULL_CHECKPOINT_AFTER_STATS = False  #@param {type:"boolean"}
TRAIN_MEAN = None
TRAIN_STD = None

CACHE_DIR = Path("data/one_val_cache")
FULL_VAL_CACHE_DIR = Path("data/full_val_cache")
CHECKPOINT_DIR = Path("checkpoints")
CASE_DIR = Path("inverse_cases")
RUNS_DIR = Path("runs_ddnm_colab")
WRAPPED_CHECKPOINT_PATH = CHECKPOINT_DIR / "wrapped_ema_checkpoint.pt"

In [ ]:
#@title 3. Imports and device
import json
import math
import urllib.parse
import urllib.request
from collections import deque

import matplotlib.pyplot as plt
import numpy as np
import torch
from tqdm.auto import tqdm
from IPython.display import Image, display
import pandas as pd

from score_training import DiffusersUNet, VPCosineSDE
from inverse.cases import CaseConfig, create_case_file
from inverse.experiment import ExperimentConfig, run_experiment

torch.manual_seed(DDNM_SEED)
np.random.seed(DDNM_SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)
if device.type == "cuda":
    print("gpu:", torch.cuda.get_device_name(0))
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    torch.backends.cudnn.benchmark = True
    torch.set_float32_matmul_precision("high")

In [ ]:
#@title 4. Download val shard and EMA checkpoint
def yandex_json(url: str) -> dict:
    req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
    with urllib.request.urlopen(req) as response:
        return json.loads(response.read().decode("utf-8"))


def yandex_list(public_key: str, path: str = "/") -> dict:
    params = {"public_key": public_key, "path": path, "limit": 1000}
    api_url = "https://cloud-api.yandex.net/v1/disk/public/resources?" + urllib.parse.urlencode(params)
    return yandex_json(api_url)


def find_public_path(public_key: str, filename: str) -> str:
    queue = deque(["/"])
    seen = set()
    while queue:
        path = queue.popleft()
        if path in seen:
            continue
        seen.add(path)
        payload = yandex_list(public_key, path)
        items = payload.get("_embedded", {}).get("items", [])
        for item in items:
            name = item.get("name", "")
            item_path = item.get("path", "")
            public_path = item_path.split(":", 1)[-1] if ":" in item_path else item_path
            if name == filename:
                return public_path
            if item.get("type") == "dir":
                queue.append(public_path)
    raise FileNotFoundError(f"Could not find {filename!r} in Yandex public folder")


def find_optional_public_path(public_key: str, filename: str):
    try:
        return find_public_path(public_key, filename)
    except FileNotFoundError:
        return None


def find_public_paths(public_key: str, predicate) -> list:
    matches = []
    queue = deque(["/"])
    seen = set()
    while queue:
        path = queue.popleft()
        if path in seen:
            continue
        seen.add(path)
        payload = yandex_list(public_key, path)
        items = payload.get("_embedded", {}).get("items", [])
        for item in items:
            name = item.get("name", "")
            item_path = item.get("path", "")
            public_path = item_path.split(":", 1)[-1] if ":" in item_path else item_path
            if item.get("type") == "dir":
                queue.append(public_path)
            elif predicate(name):
                matches.append(public_path)
    return sorted(matches)


def yandex_download_href(public_key: str, path: str) -> str:
    params = {"public_key": public_key, "path": path}
    api_url = "https://cloud-api.yandex.net/v1/disk/public/resources/download?" + urllib.parse.urlencode(params)
    return yandex_json(api_url)["href"]


def download_file(url: str, target: Path, desc: str) -> Path:
    target.parent.mkdir(parents=True, exist_ok=True)
    req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
    with urllib.request.urlopen(req) as response:
        total = int(response.headers.get("Content-Length", 0))
        if target.exists() and (total <= 0 or target.stat().st_size == total):
            print("using cached:", target)
            return target
        with target.open("wb") as handle, tqdm(total=total or None, unit="B", unit_scale=True, desc=desc) as bar:
            while True:
                chunk = response.read(1024 * 1024)
                if not chunk:
                    break
                handle.write(chunk)
                bar.update(len(chunk))
    return target


val_public_path = find_public_path(DATA_SOURCE, VAL_SHARD_NAME)
ema_public_path = find_public_path(DATA_SOURCE, EMA_CHECKPOINT_NAME)
stats_public_path = find_optional_public_path(DATA_SOURCE, STATS_FILENAME)
full_public_path = find_optional_public_path(DATA_SOURCE, FULL_CHECKPOINT_NAME)
print("val path:", val_public_path)
print("EMA checkpoint path:", ema_public_path)
print("stats JSON path:", stats_public_path)
print("full checkpoint path:", full_public_path)

val_path = CACHE_DIR / VAL_SHARD_NAME
checkpoint_path = CHECKPOINT_DIR / EMA_CHECKPOINT_NAME
stats_path = CACHE_DIR / STATS_FILENAME if stats_public_path is not None else None
full_checkpoint_path = CHECKPOINT_DIR / FULL_CHECKPOINT_NAME if full_public_path is not None else None

download_file(yandex_download_href(DATA_SOURCE, val_public_path), val_path, VAL_SHARD_NAME)
download_file(yandex_download_href(DATA_SOURCE, ema_public_path), checkpoint_path, EMA_CHECKPOINT_NAME)
if stats_public_path is not None:
    download_file(yandex_download_href(DATA_SOURCE, stats_public_path), stats_path, STATS_FILENAME)
elif DOWNLOAD_FULL_CHECKPOINT_FOR_STATS and full_public_path is not None:
    download_file(yandex_download_href(DATA_SOURCE, full_public_path), full_checkpoint_path, FULL_CHECKPOINT_NAME)
else:
    print("WARNING: no standalone stats JSON and full-checkpoint download disabled/unavailable.")

print("val shard:", val_path, f"{val_path.stat().st_size / 1024**2:.1f} MB")
print("EMA checkpoint:", checkpoint_path, f"{checkpoint_path.stat().st_size / 1024**2:.1f} MB")
if stats_path is not None and stats_path.exists():
    print("stats JSON:", stats_path, f"{stats_path.stat().st_size / 1024**2:.3f} MB")
if full_checkpoint_path is not None and full_checkpoint_path.exists():
    print("full checkpoint for stats:", full_checkpoint_path, f"{full_checkpoint_path.stat().st_size / 1024**2:.1f} MB")

In [ ]:
#@title 5. Inspect val shard and prepare mean/std
with np.load(val_path) as arrays:
    print("keys:", list(arrays.keys()))
    val_images = arrays["images"].astype(np.float32, copy=False)
print("val images:", val_images.shape, val_images.dtype)

stats_source = "manual"
if TRAIN_MEAN is not None and TRAIN_STD is not None:
    mean = np.asarray(TRAIN_MEAN, dtype=np.float32)
    std = np.asarray(TRAIN_STD, dtype=np.float32)
elif stats_path is not None and stats_path.exists():
    stats_payload = json.loads(stats_path.read_text())
    mean = np.asarray(stats_payload["mean"], dtype=np.float32)
    std = np.asarray(stats_payload["std"], dtype=np.float32)
    stats_source = str(stats_path)
elif full_checkpoint_path is not None and full_checkpoint_path.exists():
    full_payload = torch.load(full_checkpoint_path, map_location="cpu")
    if "data_stats" not in full_payload:
        raise RuntimeError(f"Full checkpoint {full_checkpoint_path} does not contain data_stats")
    stats_payload = full_payload["data_stats"]
    mean = np.asarray(stats_payload["mean"], dtype=np.float32)
    std = np.asarray(stats_payload["std"], dtype=np.float32)
    stats_source = f"{full_checkpoint_path}:data_stats"
    if DELETE_FULL_CHECKPOINT_AFTER_STATS:
        full_checkpoint_path.unlink()
        print("deleted full checkpoint after extracting stats:", full_checkpoint_path)
else:
    print("WARNING: using stats computed from val_000. Smoke test only, not final metrics.")
    mean = val_images.mean(axis=(0, 2, 3), dtype=np.float64).astype(np.float32)
    std = val_images.std(axis=(0, 2, 3), dtype=np.float64).astype(np.float32)
    stats_source = f"{val_path}:computed_val_fallback"

std = np.maximum(std, 1.0e-6)
print("normalization stats source:", stats_source)
print("mean:", mean)
print("std:", std)

In [ ]:
#@title 6. Wrap EMA-only checkpoint into the shared pipeline checkpoint format
CHANNELS = 2
IMAGE_SIZE = 64
COORDINATE_MODE = "fourier"
CHANNELS_PER_LEVEL = "96,192,384"
NUM_RES_BLOCKS = 3
ATTENTION_HEAD_DIM = 32
PADDING_MODE = "circular"
TIME_EMBEDDING_SCALE = 999.0
CLIP_PRED_X0 = 5.0

state = torch.load(checkpoint_path, map_location="cpu")
if isinstance(state, dict) and "ema_model" in state:
    state = state["ema_model"]

config = {
    "channels_per_level": CHANNELS_PER_LEVEL,
    "num_res_blocks": NUM_RES_BLOCKS,
    "attention_head_dim": ATTENTION_HEAD_DIM,
    "padding_mode": PADDING_MODE,
    "coordinate_mode": COORDINATE_MODE,
    "time_embedding_scale": TIME_EMBEDDING_SCALE,
    "clip_pred_x0": CLIP_PRED_X0,
    "sample_steps": DDNM_STEPS,
    "dropout": 0.0,
}
data_stats = {
    "mean": [float(x) for x in mean],
    "std": [float(x) for x in std],
    "channels": CHANNELS,
    "height": IMAGE_SIZE,
    "width": IMAGE_SIZE,
    "train_count": 0,
    "val_count": int(val_images.shape[0]),
    "source": str(val_path),
    "stats_source": stats_source,
}
wrapped = {
    "ema_model": state,
    "config": config,
    "data_stats": data_stats,
    "coordinate_mode": COORDINATE_MODE,
    "time_embedding_scale": TIME_EMBEDDING_SCALE,
}
WRAPPED_CHECKPOINT_PATH.parent.mkdir(parents=True, exist_ok=True)
torch.save(wrapped, WRAPPED_CHECKPOINT_PATH)
print("wrapped checkpoint:", WRAPPED_CHECKPOINT_PATH)

In [ ]:
#@title 7. Shared DDNM run helper
all_metrics_by_run = {}


def _safe_tag(value) -> str:
    return str(value).replace(".", "p").replace("-", "m")


def run_ddnm_experiment(
    operator_name: str,
    run_specs: list,
    metric_sample_count=None,
    visualization_sample_count=None,
    max_visualizations=None,
    sample_seed=None,
    visualization_seed=None,
    corruption_seed=None,
    noise_sigma=None,
    steps=None,
    soft_lambda=None,
    run_batch_size=None,
    case_batch_size=None,
    display_max_images=None,
    data_source_dir=None,
    data_tag: str = "",
    box_size: int = 32,
    downsample_factor: int = 4,
    blur_sigma: float = 2.0,
    stride: int = 4,
):
    """Build the case file and run all DDNM variants for one operator.

    Each entry in run_specs is a dict with keys:
      label       : str   — tag embedded in the output directory name
      div_weight  : float — > 0 enables Helmholtz physics projection (ddnm+physics)
      soft_lambda : float — > 0 enables soft DDNM; 0 = hard DDNM (default)
    """
    metric_sample_count = METRIC_SAMPLE_COUNT if metric_sample_count is None else metric_sample_count
    visualization_sample_count = VISUALIZATION_SAMPLE_COUNT if visualization_sample_count is None else visualization_sample_count
    max_visualizations = MAX_VISUALIZATIONS if max_visualizations is None else max_visualizations
    sample_seed = SAMPLE_SEED if sample_seed is None else sample_seed
    visualization_seed = VISUALIZATION_SEED if visualization_seed is None else visualization_seed
    corruption_seed = CORRUPTION_SEED if corruption_seed is None else corruption_seed
    noise_sigma = NOISE_SIGMA if noise_sigma is None else noise_sigma
    steps = DDNM_STEPS if steps is None else steps
    soft_lambda = SOFT_LAMBDA if soft_lambda is None else soft_lambda
    run_batch_size = RUN_BATCH_SIZE if run_batch_size is None else run_batch_size
    case_batch_size = CASE_BATCH_SIZE if case_batch_size is None else case_batch_size
    display_max_images = max_visualizations * len(run_specs) if display_max_images is None else display_max_images
    data_source_dir = CACHE_DIR if data_source_dir is None else Path(data_source_dir)

    sample_tag = "all" if int(metric_sample_count) <= 0 else str(int(metric_sample_count))
    source_suffix = f"_{data_tag}" if data_tag else ""
    case_path = (
        CASE_DIR
        / f"{operator_name}{source_suffix}_val{sample_tag}_vis{visualization_sample_count}_seed{sample_seed}.npz"
    )
    cfg = CaseConfig(
        data_source=str(data_source_dir),
        output_path=str(case_path),
        operator=operator_name,
        split="val",
        num_samples=metric_sample_count,
        sample_seed=sample_seed,
        visualization_count=visualization_sample_count,
        visualization_seed=visualization_seed,
        corruption_seed=corruption_seed,
        noise_sigma=noise_sigma,
        box_size=box_size,
        downsample_factor=downsample_factor,
        blur_sigma=blur_sigma,
        stride=stride,
        device=CASE_BUILD_DEVICE if (CASE_BUILD_DEVICE == "cpu" or torch.cuda.is_available()) else "cpu",
        case_batch_size=case_batch_size,
    )
    case_path = create_case_file(cfg)
    print("case:", case_path)

    local_metrics_paths = []
    local_output_dirs = []
    for spec in run_specs:
        div_weight = float(spec.get("div_weight", 0.0))
        spec_soft_lambda = float(spec.get("soft_lambda", soft_lambda))
        label = spec.get("label", "ddnm_phys" if div_weight > 0 else "ddnm")
        sl_tag = f"_sl{_safe_tag(spec_soft_lambda)}" if spec_soft_lambda > 0 else ""
        tag = (
            f"{label}_{operator_name}{source_suffix}"
            f"{sl_tag}_n{sample_tag}_steps{steps}"
        )
        output_dir = RUNS_DIR / tag
        exp_cfg = ExperimentConfig(
            checkpoint_path=str(WRAPPED_CHECKPOINT_PATH),
            case_file=str(case_path),
            output_dir=str(output_dir),
            method="ddnm",
            device="auto",
            batch_size=run_batch_size,
            steps=steps,
            seed=DDNM_SEED,
            max_visualizations=max_visualizations,
            visualization_sample_count=max_visualizations,
            guidance_scale=0.0,
            measurement_sigma=0.0,
            guidance_start=1.0,
            guidance_end=0.0,
            gradient_clip=0.0,
            div_weight=div_weight,
            soft_lambda=spec_soft_lambda,
            box_size=box_size,
        )
        print("running:", output_dir)
        metrics_path = run_experiment(exp_cfg)
        local_metrics_paths.append(metrics_path)
        local_output_dirs.append(output_dir)
        all_metrics_by_run[str(output_dir)] = metrics_path
        figure_count = len(list((output_dir / "figures").glob("*.png")))
        print("saved metrics:", metrics_path)
        print("saved figures:", figure_count, "in", output_dir / "figures")

    frames = []
    for path in local_metrics_paths:
        df = pd.read_csv(path)
        df["run"] = str(path.parent)
        frames.append(df)
    local_metrics = pd.concat(frames, ignore_index=True)
    local_out = RUNS_DIR / f"{operator_name}{source_suffix}_metrics.csv"
    RUNS_DIR.mkdir(parents=True, exist_ok=True)
    local_metrics.to_csv(local_out, index=False)
    group_cols = [
        col for col in ["method_variant", "method", "physics_informed", "div_weight", "operator", "run"]
        if col in local_metrics.columns
    ]
    local_summary = (
        local_metrics
        .groupby(group_cols)[["rel_l2", "rmse", "measurement_error", "divergence", "vorticity_rmse"]]
        .mean()
        .reset_index()
    )
    display(local_summary)
    print("operator csv:", local_out)

    if all_metrics_by_run:
        cumulative_frames = []
        for run_path in all_metrics_by_run.values():
            df = pd.read_csv(run_path)
            df["run"] = str(run_path.parent)
            cumulative_frames.append(df)
        cumulative_metrics = pd.concat(cumulative_frames, ignore_index=True)
        RUNS_DIR.mkdir(parents=True, exist_ok=True)
        cumulative_csv = RUNS_DIR / "all_metrics.csv"
        cumulative_metrics.to_csv(cumulative_csv, index=False)
        cumulative_group_cols = [
            col for col in ["method_variant", "method", "physics_informed", "div_weight", "operator", "run"]
            if col in cumulative_metrics.columns
        ]
        cumulative_summary = (
            cumulative_metrics
            .groupby(cumulative_group_cols)[["rel_l2", "rmse", "measurement_error", "divergence", "vorticity_rmse"]]
            .mean()
            .reset_index()
        )
        cumulative_summary_csv = RUNS_DIR / "summary.csv"
        cumulative_summary.to_csv(cumulative_summary_csv, index=False)
        print("cumulative metrics:", cumulative_csv)
        print("cumulative summary:", cumulative_summary_csv)

    figure_paths = []
    for output_dir in local_output_dirs:
        figure_paths.extend(sorted((output_dir / "figures").glob("*.png")))
    print("displaying figures:", min(len(figure_paths), display_max_images), "of", len(figure_paths))
    for path in figure_paths[:display_max_images]:
        print(path)
        display(Image(filename=str(path)))
    return local_metrics_paths

In [ ]:
#@title 8. Configure sparse-grid run
# Sparse grid: velocity known at every 4th pixel (stride 4), rest reconstructed.
SPARSE_METRIC_SAMPLE_COUNT = METRIC_SAMPLE_COUNT  #@param {type:"integer"}
SPARSE_VISUALIZATION_SAMPLE_COUNT = VISUALIZATION_SAMPLE_COUNT  #@param {type:"integer"}
SPARSE_MAX_VISUALIZATIONS = min(MAX_VISUALIZATIONS, SPARSE_VISUALIZATION_SAMPLE_COUNT)
SPARSE_RUN_BATCH_SIZE = RUN_BATCH_SIZE  #@param {type:"integer"}
SPARSE_DDNM_STEPS = DDNM_STEPS  #@param {type:"integer"}
SPARSE_CORRUPTION_SEED = CORRUPTION_SEED  #@param {type:"integer"}
SPARSE_NOISE_SIGMA = NOISE_SIGMA  #@param {type:"number"}
SPARSE_SOFT_LAMBDA = SOFT_LAMBDA  #@param {type:"number"}
SPARSE_PHYS_DIV_WEIGHT = PHYSICS_DIV_WEIGHT  #@param {type:"number"}

SPARSE_RUNS = [
    {"label": "ddnm",      "div_weight": 0.0,                    "soft_lambda": SPARSE_SOFT_LAMBDA},
    {"label": "ddnm_phys", "div_weight": SPARSE_PHYS_DIV_WEIGHT, "soft_lambda": SPARSE_SOFT_LAMBDA},
]
SPARSE_RUNS

In [ ]:
#@title 9. Run sparse-grid DDNM experiments
sparse_metrics_paths = run_ddnm_experiment(
    "sparse_grid",
    SPARSE_RUNS,
    metric_sample_count=SPARSE_METRIC_SAMPLE_COUNT,
    visualization_sample_count=SPARSE_VISUALIZATION_SAMPLE_COUNT,
    max_visualizations=SPARSE_MAX_VISUALIZATIONS,
    corruption_seed=SPARSE_CORRUPTION_SEED,
    noise_sigma=SPARSE_NOISE_SIGMA,
    steps=SPARSE_DDNM_STEPS,
    run_batch_size=SPARSE_RUN_BATCH_SIZE,
)

In [ ]:
#@title 10. Configure box-mask run
# Box mask: velocity known inside central 32×32 window, outer ring reconstructed.
BOX_METRIC_SAMPLE_COUNT = METRIC_SAMPLE_COUNT  #@param {type:"integer"}
BOX_VISUALIZATION_SAMPLE_COUNT = VISUALIZATION_SAMPLE_COUNT  #@param {type:"integer"}
BOX_MAX_VISUALIZATIONS = min(MAX_VISUALIZATIONS, BOX_VISUALIZATION_SAMPLE_COUNT)
BOX_RUN_BATCH_SIZE = RUN_BATCH_SIZE  #@param {type:"integer"}
BOX_DDNM_STEPS = DDNM_STEPS  #@param {type:"integer"}
BOX_CORRUPTION_SEED = CORRUPTION_SEED  #@param {type:"integer"}
BOX_NOISE_SIGMA = NOISE_SIGMA  #@param {type:"number"}
BOX_SIZE = 32  #@param {type:"integer"}
BOX_SOFT_LAMBDA = SOFT_LAMBDA  #@param {type:"number"}
BOX_PHYS_DIV_WEIGHT = PHYSICS_DIV_WEIGHT  #@param {type:"number"}

BOX_RUNS = [
    {"label": "ddnm",      "div_weight": 0.0,                 "soft_lambda": BOX_SOFT_LAMBDA},
    {"label": "ddnm_phys", "div_weight": BOX_PHYS_DIV_WEIGHT, "soft_lambda": BOX_SOFT_LAMBDA},
]
BOX_RUNS

In [ ]:
#@title 11. Run box-mask DDNM experiments
box_metrics_paths = run_ddnm_experiment(
    "box_mask",
    BOX_RUNS,
    metric_sample_count=BOX_METRIC_SAMPLE_COUNT,
    visualization_sample_count=BOX_VISUALIZATION_SAMPLE_COUNT,
    max_visualizations=BOX_MAX_VISUALIZATIONS,
    corruption_seed=BOX_CORRUPTION_SEED,
    noise_sigma=BOX_NOISE_SIGMA,
    box_size=BOX_SIZE,
    steps=BOX_DDNM_STEPS,
    run_batch_size=BOX_RUN_BATCH_SIZE,
)

In [ ]:
#@title 12. Configure downsample run
# Downsample: average-pool 64×64 → 16×16, then reconstruct full resolution.
DS_METRIC_SAMPLE_COUNT = METRIC_SAMPLE_COUNT  #@param {type:"integer"}
DS_VISUALIZATION_SAMPLE_COUNT = VISUALIZATION_SAMPLE_COUNT  #@param {type:"integer"}
DS_MAX_VISUALIZATIONS = min(MAX_VISUALIZATIONS, DS_VISUALIZATION_SAMPLE_COUNT)
DS_RUN_BATCH_SIZE = RUN_BATCH_SIZE  #@param {type:"integer"}
DS_DDNM_STEPS = DDNM_STEPS  #@param {type:"integer"}
DS_CORRUPTION_SEED = CORRUPTION_SEED  #@param {type:"integer"}
DS_NOISE_SIGMA = NOISE_SIGMA  #@param {type:"number"}
DS_DOWNSAMPLE_FACTOR = 4  #@param {type:"integer"}
DS_SOFT_LAMBDA = SOFT_LAMBDA  #@param {type:"number"}
DS_PHYS_DIV_WEIGHT = PHYSICS_DIV_WEIGHT  #@param {type:"number"}

DS_RUNS = [
    {"label": "ddnm",      "div_weight": 0.0,               "soft_lambda": DS_SOFT_LAMBDA},
    {"label": "ddnm_phys", "div_weight": DS_PHYS_DIV_WEIGHT, "soft_lambda": DS_SOFT_LAMBDA},
]
DS_RUNS

In [ ]:
#@title 13. Run downsample DDNM experiments
ds_metrics_paths = run_ddnm_experiment(
    "downsample",
    DS_RUNS,
    metric_sample_count=DS_METRIC_SAMPLE_COUNT,
    visualization_sample_count=DS_VISUALIZATION_SAMPLE_COUNT,
    max_visualizations=DS_MAX_VISUALIZATIONS,
    corruption_seed=DS_CORRUPTION_SEED,
    noise_sigma=DS_NOISE_SIGMA,
    downsample_factor=DS_DOWNSAMPLE_FACTOR,
    steps=DS_DDNM_STEPS,
    run_batch_size=DS_RUN_BATCH_SIZE,
)

In [ ]:
#@title 14. Configure blur run
# Periodic Gaussian blur (σ=2) with observation noise; Wiener deconvolution pinv.
BLUR_METRIC_SAMPLE_COUNT = METRIC_SAMPLE_COUNT  #@param {type:"integer"}
BLUR_VISUALIZATION_SAMPLE_COUNT = VISUALIZATION_SAMPLE_COUNT  #@param {type:"integer"}
BLUR_MAX_VISUALIZATIONS = min(MAX_VISUALIZATIONS, BLUR_VISUALIZATION_SAMPLE_COUNT)
BLUR_RUN_BATCH_SIZE = RUN_BATCH_SIZE  #@param {type:"integer"}
BLUR_DDNM_STEPS = DDNM_STEPS  #@param {type:"integer"}
BLUR_CORRUPTION_SEED = CORRUPTION_SEED  #@param {type:"integer"}
BLUR_SIGMA = 2.0  #@param {type:"number"}
# Use nonzero noise for blur to test robustness; set to 0.0 for noiseless deconvolution.
BLUR_NOISE_SIGMA_RUN = BLUR_NOISE_SIGMA  #@param {type:"number"}
# Soft DDNM (λ>0) can be more stable for noisy blur; hard DDNM (λ=0) uses Wiener pinv directly.
BLUR_SOFT_LAMBDA = SOFT_LAMBDA  #@param {type:"number"}
BLUR_PHYS_DIV_WEIGHT = PHYSICS_DIV_WEIGHT  #@param {type:"number"}

BLUR_RUNS = [
    {"label": "ddnm",      "div_weight": 0.0,                 "soft_lambda": BLUR_SOFT_LAMBDA},
    {"label": "ddnm_phys", "div_weight": BLUR_PHYS_DIV_WEIGHT, "soft_lambda": BLUR_SOFT_LAMBDA},
]
BLUR_RUNS

In [ ]:
#@title 15. Run blur DDNM experiments
blur_metrics_paths = run_ddnm_experiment(
    "blur",
    BLUR_RUNS,
    metric_sample_count=BLUR_METRIC_SAMPLE_COUNT,
    visualization_sample_count=BLUR_VISUALIZATION_SAMPLE_COUNT,
    max_visualizations=BLUR_MAX_VISUALIZATIONS,
    corruption_seed=BLUR_CORRUPTION_SEED,
    noise_sigma=BLUR_NOISE_SIGMA_RUN,
    blur_sigma=BLUR_SIGMA,
    steps=BLUR_DDNM_STEPS,
    run_batch_size=BLUR_RUN_BATCH_SIZE,
    soft_lambda=BLUR_SOFT_LAMBDA,
)

In [ ]:
#@title 16. (Optional) Full val evaluation on all val_*.npz shards
RUN_FULL_VAL_EVAL = False  #@param {type:"boolean"}
FULL_VAL_RUN_BATCH_SIZE = RUN_BATCH_SIZE * 4  #@param {type:"integer"}
FULL_VAL_METRIC_SAMPLE_COUNT = FULL_VAL_RUN_BATCH_SIZE * 5  #@param {type:"integer"}
FULL_VAL_DDNM_STEPS = DDNM_STEPS  #@param {type:"integer"}
FULL_VAL_VISUALIZATION_SAMPLE_COUNT = 0  #@param {type:"integer"}
FULL_VAL_MAX_VISUALIZATIONS = 0  #@param {type:"integer"}
FULL_VAL_DISPLAY_MAX_IMAGES = 0  #@param {type:"integer"}
FULL_VAL_OPERATORS = ["sparse_grid", "box_mask", "downsample", "blur"]

if RUN_FULL_VAL_EVAL:
    val_public_paths = find_public_paths(DATA_SOURCE, lambda name: name.startswith("val_") and name.endswith(".npz"))
    if not val_public_paths:
        print("WARNING: no val_*.npz files found; using the single val shard only.")
        val_public_paths = [val_public_path]
    print("full-val shards:", len(val_public_paths))
    FULL_VAL_CACHE_DIR.mkdir(parents=True, exist_ok=True)
    for public_path in val_public_paths:
        filename = Path(public_path).name
        target = FULL_VAL_CACHE_DIR / filename
        download_file(yandex_download_href(DATA_SOURCE, public_path), target, filename)

    full_val_jobs = {
        "sparse_grid": (SPARSE_RUNS, SPARSE_CORRUPTION_SEED, SPARSE_NOISE_SIGMA, {}),
        "box_mask":    (BOX_RUNS,    BOX_CORRUPTION_SEED,    BOX_NOISE_SIGMA,    {"box_size": BOX_SIZE}),
        "downsample":  (DS_RUNS,     DS_CORRUPTION_SEED,     DS_NOISE_SIGMA,     {"downsample_factor": DS_DOWNSAMPLE_FACTOR}),
        "blur":        (BLUR_RUNS,   BLUR_CORRUPTION_SEED,   BLUR_NOISE_SIGMA_RUN, {"blur_sigma": BLUR_SIGMA, "soft_lambda": BLUR_SOFT_LAMBDA}),
    }
    full_val_metrics_paths = []
    for operator_name in FULL_VAL_OPERATORS:
        run_specs, corruption_seed, noise_sigma, extra_kwargs = full_val_jobs[operator_name]
        full_val_metrics_paths.extend(run_ddnm_experiment(
            operator_name,
            run_specs,
            metric_sample_count=FULL_VAL_METRIC_SAMPLE_COUNT,
            visualization_sample_count=FULL_VAL_VISUALIZATION_SAMPLE_COUNT,
            max_visualizations=FULL_VAL_MAX_VISUALIZATIONS,
            corruption_seed=corruption_seed,
            noise_sigma=noise_sigma,
            steps=FULL_VAL_DDNM_STEPS,
            run_batch_size=FULL_VAL_RUN_BATCH_SIZE,
            data_source_dir=FULL_VAL_CACHE_DIR,
            data_tag="full_val",
            display_max_images=FULL_VAL_DISPLAY_MAX_IMAGES,
            **extra_kwargs,
        ))
else:
    print("Skipping full-val eval. Set RUN_FULL_VAL_EVAL=True for final metrics on all val_*.npz shards.")

In [ ]:
#@title 17. Final combined CSV across all observation operators
if not all_metrics_by_run:
    raise RuntimeError("No metrics produced yet. Run at least one observation cell first.")

frames = []
for path in all_metrics_by_run.values():
    df = pd.read_csv(path)
    df["run"] = str(path.parent)
    frames.append(df)
all_metrics = pd.concat(frames, ignore_index=True)
RUNS_DIR.mkdir(parents=True, exist_ok=True)
all_csv = RUNS_DIR / "all_metrics.csv"
all_metrics.to_csv(all_csv, index=False)

group_cols = [
    col for col in ["method_variant", "method", "physics_informed", "div_weight", "operator", "run"]
    if col in all_metrics.columns
]
summary = (
    all_metrics
    .groupby(group_cols)[["rel_l2", "rmse", "measurement_error", "divergence", "vorticity_rmse"]]
    .mean()
    .reset_index()
)
summary_csv = RUNS_DIR / "summary.csv"
summary.to_csv(summary_csv, index=False)
display(summary)
print("all metrics:", all_csv)
print("summary:", summary_csv)

## Parameter guide

Each observation operator has a configure cell and a run cell. Adjust the configure cell, then execute the run cell.

### Core settings (cell 2)
| Parameter | Default | Meaning |
|---|---|---|
| `METRIC_SAMPLE_COUNT` | 16 | Validation samples used for metrics per run |
| `VISUALIZATION_SAMPLE_COUNT` | 8 | Stable visualization prefix size (fixed by `VISUALIZATION_SEED`) |
| `MAX_VISUALIZATIONS` | 8 | PNGs saved and displayed per run; keep ≤ `VISUALIZATION_SAMPLE_COUNT` |
| `DDNM_STEPS` | 128 | Diffusion denoising steps; use 64 for debug, 256 for final |
| `SOFT_LAMBDA` | 0.0 | λ=0 → hard DDNM; λ>0 → soft DDNM (more stable for noisy observations) |
| `BLUR_NOISE_SIGMA` | 0.05 | Observation noise added to blurred field |
| `RUN_BATCH_SIZE` | 32 | GPU batch size; reduce to 16 or 8 if CUDA OOM |
| `PHYSICS_DIV_WEIGHT` | 1.0 | Any value > 0 enables Helmholtz projection after data correction |

### DDNM algorithm

On each reverse step:
1. The score model predicts `ε_θ(x_t, t)` → compute `x̂₀ = (x_t − σ_t ε_θ) / μ_t`.
2. Data-consistency correction:
   - **Hard DDNM** (`soft_lambda=0`): `x̂₀^corr = A†y + (I − A†A) x̂₀`
   - **Soft DDNM** (`soft_lambda>0`): `x̂₀^corr = x̂₀ + λ A†(y − A x̂₀)`
   - For mask operators both reduce to: `x̂₀^corr = M ⊙ y + (1−M) ⊙ x̂₀`
3. DDIM step: `x_{t-1} = μ_{t-1} x̂₀^corr + σ_{t-1} ε_θ`.

### Physics variant
When `div_weight > 0`, step 2 is followed by Helmholtz FFT projection of `x̂₀^corr`:
`x̂₀^phys = P_divfree(x̂₀^corr)`.  For mask operators the known pixels (`M ⊙ y`) are
re-pasted so observed values are never modified.

### Pseudoinverse by operator
| Operator | A†y formula |
|---|---|
| sparse_grid / box_mask | `M ⊙ y` (copy known pixels) |
| downsample ×4 | nearest-neighbor upsample 16×16 → 64×64 |
| blur (periodic Gaussian) | Wiener deconvolution in Fourier space: `H*(k) / (|H(k)|² + reg) · Y(k)` |

### Operators
| Method | sparse_grid | box_mask | downsample | blur |
|---|---|---|---|---|
| DDNM | ✓ primary | ✓ primary | ✓ | ✓ |
| DDNM + physics | ✓ | ✓ | ✓ | ✓ |

### Output layout
```
runs_ddnm_colab/
  ddnm_sparse_grid_n16_steps128/
    metrics.csv
    figures/
    reconstructions/x_hat_raw.pt
    run_config.json
  ddnm_phys_sparse_grid_n16_steps128/
  ddnm_box_mask_n16_steps128/
  ddnm_phys_box_mask_n16_steps128/
  ddnm_downsample_n16_steps128/
  ddnm_phys_downsample_n16_steps128/
  ddnm_blur_n16_steps128/
  ddnm_phys_blur_n16_steps128/
  sparse_grid_metrics.csv
  box_mask_metrics.csv
  downsample_metrics.csv
  blur_metrics.csv
  all_metrics.csv
  summary.csv
```

CSV columns: `sample_id, seed, method, method_variant, physics_informed, div_weight, operator, noise_sigma, rel_l2, rmse, measurement_error, divergence, vorticity_rmse`